In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go
import matplotlib.pyplot as plt

## Table of Contents

The data cleaning is divided into sections determining who fits the classification for inclusion in the superager analysis at:
1. [Timepoint 1](#timepoint1) - Participants with all data (age, education, MRI, neuropsych) at tp1 and 2 who fit the classification for inclusion in the superager analysis based off of their neuropsych data from tp1

2. [Timepoint 2](#timepoint2) - Participants with all data at tp1 and 2 who fit the classification for inclusion in the superager analysis based off of their neuropsych data from tp2. 

3. [Timepoint 1 and 2 together](#timepoint1and2) - Participants with all data at tp1 and 2 who fit the classification for inclusion in the superager analysis at tp1 and tp2


<a id='timepoint1'></a>
### Timepoint 1

In [ ]:
# Read BBHI main data
m_df = pd.read_csv("~/Documents/2023:2024/Data/BBHI/BBHI Data Timept1 NPS.csv")

print(f"Number of participants: {len(m_df)}")

# Read BBHI education data
df_edu = pd.read_excel("~/Documents/2023:2024/Data/BBHI/BBHI_YoE.xlsx")

In [ ]:
# Calculate ages at baseline to filter df for 60+ yrs

def calculate_age(birth_date: str, collection_date: str) -> int:
    """Calculate age at baseline.

    Args:
        birth_date (str): Date of birth in format mm/dd/yyyy
        collection_date (str): Date of data collection in format mm/dd/yyyy

    Returns:
        age (int): Age at assessment
    """
    birth_date = datetime.strptime(birth_date, "%m/%d/%Y")
    collection_date = datetime.strptime(collection_date, "%m/%d/%Y")

    age = collection_date.year - birth_date.year

    # Check if the birthday has occurred for the current year
    if (collection_date.month, collection_date.day) < (
        birth_date.month,
        birth_date.day,
    ):
        age -= 1

    return age


# 'Q1 GNRL Age' is DOB and 'w1_nps_date' is data collection date
# Adds new column to df with age
m_df["age"] = m_df.apply(
    lambda row: calculate_age(row["Q1 GNRL Age"], row["w1_nps_date"]), axis=1
)

# Manually check that everything is running correctly
m_df[["id", "age", "Q1 GNRL Age", "w1_nps_date"]].head(5)

In [ ]:
# Filter df to only include 60+yrs

# Print the number of participants before filtering
print(f"Number of participants before filtering: {len(m_df)}")

agefiltered_df = m_df[m_df["age"] >= 60]

# Print the number of participants after filtering
print(f"Number of participants after filtering: {len(agefiltered_df)}")

agefiltered_df[["id", "age"]].head(5)

In [ ]:
# Read BBHI MRI data timept 1
df_MRI1 = pd.read_csv("~/Documents/2023:2024/Data/BBHI/MRI_info_TP1.csv")

# Create a new variable to see who has MRI data
mask = (
    (df_MRI1["mri_1_T1_discarded"] != "discarded")
    & (df_MRI1["mri_1_T2_discarded"] != "discarded")
    & (df_MRI1["mri_1_T1_availability"] > 0)
    & (df_MRI1["mri_1_T2_availability"] > 0)
    & (df_MRI1["mri_1_resting_duration"] == 10)  # This is because some of the first MRI scans were shorter and need to be excluded from tp1
    & (df_MRI1["mri_1_resting_availability"] == 1)
)

df_MRI1["mri_data"] = np.where(mask, 1, 0)

df_MRI1.sample(10)[
    [
        "mri_1_T1_discarded",
        "mri_1_T2_discarded",
        "mri_1_T2_availability",
        "mri_1_resting_duration",
        "mri_1_T1_availability",
        "mri_1_resting_availability",
        "mri_data",
    ]
]

In [ ]:
# Read BBHI MRI data timept 2
df_MRI2 = pd.read_csv("~/Documents/2023:2024/Data/BBHI/MRI_info_TP2.csv")

# Remove the sub- in the id (id numbers are formatted with sub-number in this csv only)
df_MRI2["id"] = df_MRI2["id"].str.replace("sub-", "").astype(int)

# Create a new variable to see who has MRI data
mask = (
    (df_MRI2["mri_2_T1_discarded"] != "discarded")
    & (df_MRI2["mri_2_T2_discarded"] != "discarded")
    & (df_MRI2["mri_2_T1_availability"] > 0)
    & (df_MRI2["mri_2_T2_availability"] > 0)
    & (df_MRI2["mri_2_resting_availability"] == 1)
)

df_MRI2["mri_data_tp2"] = np.where(mask, 1, 0)

df_MRI2.sample(10)[
    [
        "mri_2_T1_discarded",
        "mri_2_T2_discarded",
        "mri_2_T1_availability",
        "mri_2_T2_availability",
        "mri_2_resting_availability",
        "mri_data_tp2",
    ]
]

In [ ]:
# Merge dfs by participant ID for main data and education
merged_df = pd.merge(agefiltered_df, df_edu, on="id", how="inner")

# Print the number of participants after filtering
print(f"Number of participants: {len(merged_df)}")

In [ ]:
# Merge dfs by participant ID for main data and MRI timept 1
merged_df = pd.merge(merged_df, df_MRI1, on="id", how="inner")

# Print the number of participants after filtering
print(f"Number of participants: {len(merged_df)}")

In [ ]:
# Merge dfs by participant ID for main data and MRI timept 2
merged_df = pd.merge(merged_df, df_MRI2, on="id", how="inner")

# Print the number of participants after filtering
print(f"Number of participants: {len(merged_df)}")

In [ ]:
# Filter df to only include participants with MRI data

# Print the number of participants before filtering
print(f"Number of participants before filtering: {len(merged_df)}")

filtered_df = merged_df[(merged_df["mri_data"] == 1) & (merged_df["mri_data_tp2"] == 1)]

# Print the number of participants after filtering
print(f"Number of participants after filtering: {len(filtered_df)}")

filtered_df[["id", "mri_data", "mri_data_tp2"]].head(5)

In [ ]:
# Read BBHI timepoint 2 neuropsych data
np_tp2_df = pd.read_csv("~/Documents/2023:2024/Data/BBHI/BBHI Data Timept2 NPS.csv")

print(f"Number of participants: {len(np_tp2_df)}")

In [ ]:
# Merge dfs by participant ID for neuropsych tp2 data and filtered df
filtered_df = pd.merge(filtered_df, np_tp2_df, on="id", how="inner")

# Print the number of participants after filtering
print(f"Number of participants: {len(filtered_df)}")

In [ ]:
# Drop participants with missing neuropsych data and -1 values in the specific columns

# Get the number of rows before dropping participants
rows_before = filtered_df.shape[0]

columns = [
    "age",  # Age
    "YoE",  # Years of education
    "w1_delayed_recall_raw",  # RAVLT tp1
    "w1_sem_fluency_raw",  # Semantic fluency tp1
    "w1_tmt_b_raw",  # TMT B tp1
    "w1_tmt_a_raw",  # TMT A tp1
    "w1_direct_digits_raw",  # Digit span forward tp1
    "w1_inverse_digits_raw",  # Digit span backward tp1
    "delayed_recall_raw",  # RAVLT tp2 
    "sem_fluency_raw",  # Semantic fluency tp2 
    "tmt_b_raw",  # TMT B tp2 
    "tmt_a_raw",  # TMT A tp2 
    "direct_digits_raw",  # Digit span forward tp2 
    "inverse_digits_raw",  # Digit span backward tp2 
]

# Drop participants with missing neuropsych data
filtered_df.dropna(subset=columns, inplace=True)

# Drop participants with -1 values in specific columns
for col in columns:
    filtered_df = filtered_df[filtered_df[col] != -1]

# Get the number of rows after dropping participants
rows_after = filtered_df.shape[0]

# Calculate the number of participants dropped
participants_dropped = rows_before - rows_after

participants_dropped

In [ ]:
# Look at data for available longitudinal analysis

participants = filtered_df["age"].count()
mean_age = filtered_df["age"].mean()
std_dev_age = filtered_df["age"].std()
min_age = filtered_df["age"].min()
max_age = filtered_df["age"].max()

print(f"Number of participants: {participants}")
print(f"Mean age: {mean_age:.2f}")
print(f"SD age: {std_dev_age:.2f}")
print(f"Range age: {min_age:.2f} - {max_age:.2f}")

The following is a calculation based on the criteria proposed by [Sun et al. (2016)](https://pubmed.ncbi.nlm.nih.gov/27629716/) on who can be included in a superager analyses and who is a superager. Our criteria are a slight variation as follows:

- All participants must:
  - Be age 60+
- Control participants must:
  - Score within 1.5 SD of the norm for age and education on the TMT A and B, semantic fluency, digit span forward and backward based on the neuronorma data from [Peña-Casanova et al. (2009a)](https://pubmed.ncbi.nlm.nih.gov/19661109/) and [Peña-Casanova et al. (2009b)](https://pubmed.ncbi.nlm.nih.gov/19648583/) with Spanish adults.
- Superagers must:
  - Score at or above the mean for age 16-29 year olds on the RAVLT long delay free recall based on normative data from [Schmidt (1996)](https://scholar.google.co.uk/scholar?hl=en&as_sdt=0%2C5&q=Schmidt%2C+M.+%281996%29.+Rey+Auditory+and+Verbal+Learning+Test%3A+A+handbook.+Los+Angeles%2C+CA%3A+Western+Psychological+Services&btnG=)
  - Score above 1 SD below the norm for age and education on the TMT B based on the neuronorma data 
  - Score above 1.5 SD below the norm for age and education on the TMT A and B, semantic fluency, digit span forward and backward based on the neuronorma data 

In [ ]:
# Readin the Neuronorma data (from the two publication above) in excel form & put into a df

xls = pd.ExcelFile(
    "/Users/rachelmorse/SuperAgers/Neuronorma data TMT, SDMT, DS, SF.xlsx"
)
score_mappings = {
    sheet_name: pd.read_excel(xls, sheet_name) for sheet_name in xls.sheet_names
}

In [ ]:
# Round down years of education data because some participants have data that is not a whole number (e.g. YoE = 9.5)
# This gives participants the lower education value because they will have their cognitive test scores normalized according to education level, disadvantaging them if it is rounded up

filtered_df["YoE"] = np.floor(filtered_df["YoE"])

To determine who is within 1.5 SD of the norm, first the raw scores from the neuropsychological tests must be transformed into scaled scores based on age and then education. 

In [ ]:
# Create scaled scores for TMT-A that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw TMT-A scores to scaled TMT-A scores based on age.

    Args:
        raw_score (int): Raw TMT-A score from filtered_df
        age (int): Age of participant
        TMTA (int): Raw TMT-A score from Neuronorma data
        Scale Score (int): Scaled TMT-A score from Neuronorma data

    Returns:
        TMTA_norm_age (int): Scaled TMT-A score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["TMTA"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


filtered_df["TMTA_norm_age"] = filtered_df.apply(
    lambda row: map_raw_to_scaled(row["w1_tmt_a_raw"], row["age"]), axis=1
)

filtered_df[["id", "age", "w1_tmt_a_raw", "TMTA_norm_age"]].head(10)

In [ ]:
# Create scaled scores for TMT-A that adjust now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="TMTA",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
TMTA_norm_list = []

for _, row in filtered_df.iterrows():
    tmta_norm_age = row["TMTA_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    TMTA_norm = df_edu_data.loc[tmta_norm_age, YoE]
    TMTA_norm_list.append(TMTA_norm)

filtered_df["TMTA_norm"] = TMTA_norm_list

filtered_df[["id", "YoE", "TMTA_norm_age", "TMTA_norm"]].head(10)

In [ ]:
# Create scaled scores for TMT-B that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw TMT-B scores to scaled TMT-B scores based on age.

    Args:
        raw_score (int): Raw TMT-B score from filtered_df
        age (int): Age of participant
        TMTB (int): Raw TMT-B score from Neuronorma data
        Scale Score (int): Scaled TMT-B score from Neuronorma data

    Returns:
        TMTB_norm_age (int): Scaled TMT-B score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["TMTB"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


filtered_df["TMTB_norm_age"] = filtered_df.apply(
    lambda row: map_raw_to_scaled(row["w1_tmt_b_raw"], row["age"]), axis=1
)

filtered_df[["id", "age", "w1_tmt_b_raw", "TMTB_norm_age"]].head(10)

In [ ]:
# Create scaled scores for TMT-B that adjust now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="TMTB",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
TMTB_norm_list = []

for index, row in filtered_df.iterrows():
    tmtb_norm_age = row["TMTB_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    TMTB_norm = df_edu_data.loc[tmtb_norm_age, YoE]
    TMTB_norm_list.append(TMTB_norm)  # Corrected here

filtered_df["TMTB_norm"] = TMTB_norm_list

filtered_df[["id", "YoE", "TMTB_norm_age", "TMTB_norm"]].head(10)

In [ ]:
# Create scaled scores for Digit Span - forward

# Start by rounding up Digit Span - forward score because some participants have data that is not a whole number (e.g. w1_direct_digits_raw = 4.5)
filtered_df["w1_direct_digits_raw"] = np.ceil(filtered_df["w1_direct_digits_raw"])

In [ ]:
# Create scaled scores for Digit Span - forward that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw DS-F scores to scaled DS-F scores based on age.

    Args:
        raw_score (int): Raw DS-F score from filtered_df
        age (int): Age of participant
        TMTA (int): Raw DS-F score from Neuronorma data
        Scale Score (int): Scaled DS-F score from Neuronorma data

    Returns:
        dsf_norm_age (int): Scaled DS-F score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["DS_F"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


filtered_df["dsf_norm_age"] = filtered_df.apply(
    lambda row: map_raw_to_scaled(row["w1_direct_digits_raw"], row["age"]), axis=1
)

filtered_df.sample(10)[["id", "age", "w1_direct_digits_raw", "dsf_norm_age"]].head(10)

In [ ]:
# Create scaled scores for DS-Forward that adjusts now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="DS_F",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
dsf_norm_list = []

for index, row in filtered_df.iterrows():
    dsf_norm_age = row["dsf_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    dsf_norm = df_edu_data.loc[dsf_norm_age, YoE]
    dsf_norm_list.append(dsf_norm)  # Corrected here

filtered_df["dsf_norm"] = dsf_norm_list

# Display a random 10 rows
filtered_df.sample(10)[["id", "YoE", "dsf_norm_age", "dsf_norm"]]

In [ ]:
# Create scaled scores for Digit Span - backward

# Start by rounding up Digit Span - backward score because some participants have data that is not a whole number (e.g. w1_direct_digits_raw = 2.5)
filtered_df["w1_inverse_digits_raw"] = np.ceil(filtered_df["w1_inverse_digits_raw"])

In [ ]:
# Create scaled scores for Digit Span - backward adjusted for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw DS-B scores to scaled DS-B scores based on age.

    Args:
        raw_score (int): Raw DS-B score from filtered_df
        age (int): Age of participant
        TMTA (int): Raw DS-B score from Neuronorma data
        Scale Score (int): Scaled DS-B score from Neuronorma data

    Returns:
        dsf_norm_age (int): Scaled DS-B score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["DS_B"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


filtered_df["dsb_norm_age"] = filtered_df.apply(
    lambda row: map_raw_to_scaled(row["w1_inverse_digits_raw"], row["age"]), axis=1
)

filtered_df.sample(10)[["id", "age", "w1_inverse_digits_raw", "dsb_norm_age"]].head(10)

In [ ]:
# Create scaled scores for DS-Backward that adjusts now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="DS_B",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
dsb_norm_list = []

for index, row in filtered_df.iterrows():
    dsb_norm_age = row["dsb_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    dsb_norm = df_edu_data.loc[dsb_norm_age, YoE]
    dsb_norm_list.append(dsb_norm)  # Corrected here

filtered_df["dsb_norm"] = dsb_norm_list

# Display a random 10 rows
filtered_df.sample(10)[["id", "YoE", "dsb_norm_age", "dsb_norm"]]

In [ ]:
# Create scaled scores for Semantic Fluency that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw SF scores to scaled SF scores based on age.

    Args:
        raw_score (int): Raw SF score from filtered_df
        age (int): Age of participant
        TMTA (int): Raw SF score from Neuronorma data
        Scale Score (int): Scaled SF score from Neuronorma data

    Returns:
        dsf_norm_age (int): Scaled SF score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["SF"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


filtered_df["sf_norm_age"] = filtered_df.apply(
    lambda row: map_raw_to_scaled(row["w1_sem_fluency_raw"], row["age"]), axis=1
)

filtered_df.sample(10)[["id", "age", "w1_sem_fluency_raw", "sf_norm_age"]].head(10)

In [ ]:
# Create scaled scores for semantic fluency that adjusts for now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="SF",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
sf_norm_list = []

for index, row in filtered_df.iterrows():
    sf_norm_age = row["sf_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    sf_norm = df_edu_data.loc[sf_norm_age, YoE]
    sf_norm_list.append(sf_norm)  # Corrected here

filtered_df["sf_norm"] = sf_norm_list

# Display a random 10 rows
filtered_df.sample(10)[["id", "YoE", "sf_norm_age", "sf_norm"]]

In [ ]:
# Calculate who is superager based off of RAVLT score

# Schmidt 1996 - age 16-29 RAVLT-Delayed recall is scoring 12+ (no data adjusted by sex)

filtered_df.loc[filtered_df["w1_delayed_recall_raw"] >= 12, "superager_RAVLT"] = 1
filtered_df.loc[filtered_df["w1_delayed_recall_raw"] < 12, "superager_RAVLT"] = 0

print("Number of superagers by RAVLT criteria:")
print(filtered_df[filtered_df["superager_RAVLT"] == 1]["id"].count())

# Manually check that everything is running correctly
filtered_df[["id", "w1_delayed_recall_raw", "superager_RAVLT"]].head(10)

With the scaled scores calculated, the mean is 10 and the SD is 3 for all variables. For the Neuronorma data, see [Peña-Casanova et al. (2009c)](https://pubmed.ncbi.nlm.nih.gov/19549723/) for more info. 

In [ ]:
# Create a superager variable that = 1 when superagers are above 1SD below the norm for TMT-B and meet the RAVLT criteria

# Define the variables and the lower bound
variables = ["TMTB_norm"]
lower_bound = 10 - 1 * 3  # Scaled score of 10 is the mean and SD is 3 for all variables

# Create new column 'superager' and initialize it to 0
filtered_df["superager"] = 0

# Update 'superager' to 1 for participants who score above the lower bound for TMT-B and have 1 for 'superager_RAVLT'
filtered_df.loc[
    (filtered_df[variables] >= lower_bound).all(axis=1) & (filtered_df["superager_RAVLT"] == 1), 
    "superager"
] = 1

# Display a random sample of 10 rows
sample_df = filtered_df.sample(10)

print(filtered_df[filtered_df["superager"] == 1]["id"].count())

# Display the relevant columns
relevant_columns = ["id", "age", "YoE","superager"]
sample_df[relevant_columns]


In [ ]:
# Create new variables that = 1 when non-superager participants are within 1.5SD of the norm and = 1 when superager participants are above 1.5SD below the norm

variables = ["sf", "dsf", "TMTA", "TMTB", "dsb"]

for var in variables:
    # Calculate lower and upper bounds for each variable
    lower_bound = 10 - 1.5 * 3  # Scaled score of 10 is the mean and SD is 3 for all variables
    upper_bound = 10 + 1.5 * 3

    # Create new column 'normSD_x' where x is the variable
    filtered_df[f"normSD_{var}"] = 0

    # Update 'normSD_x' to 1 if non-superager participants are within the bounds for all the relevant variables 
    filtered_df.loc[
        (filtered_df[f"{var}_norm"] >= lower_bound) 
        & (filtered_df[f"{var}_norm"] <= upper_bound)
        & (filtered_df["superager"] == 0), 
        f"normSD_{var}"
    ] = 1

    # Update 'normSD_x' to 1 if all superager participants meet the lower bound for all the relevant variables 
    filtered_df.loc[
        (filtered_df[f"{var}_norm"] >= lower_bound)
        & (filtered_df["superager"] == 1), 
        f"normSD_{var}"
    ] = 1

# Display a random sample of 10 rows
sample_df = filtered_df.sample(10)

# Display the relevant columns
relevant_columns = ["id", "age", "YoE"] + [f"normSD_{var}" for var in variables]
sample_df[relevant_columns]


In [ ]:
# Calculate who is within the norm for RAVLT

# Define the means and standard deviations for each age group
mean_sd = {
    '60-69': {'mean': 8.8, 'sd': 3.0},  # mean and SD for age group 60-69
    '70+': {'mean': 7.0, 'sd': 2.4}  # mean and SD for age group 70+
}

# Create a new column 'age_group'
filtered_df['age_group'] = pd.cut(filtered_df['age'], bins=[59, 69, np.inf], labels=['60-69', '70+'])

# Create new column 'normSD_var'
filtered_df["normSD_RAVLT"] = 0

# For each age group, update 'normSD_var' based on the bounds for that group
for age_group, params in mean_sd.items():
    lower_bound = params['mean'] - 1.5 * params['sd']
    upper_bound = params['mean'] + 1.5 * params['sd']

    filtered_df.loc[
        (filtered_df['age_group'] == age_group) 
        & (filtered_df['w1_delayed_recall_raw'] >= lower_bound) 
        & (filtered_df['w1_delayed_recall_raw'] <= upper_bound)
        & (filtered_df["superager"] == 0), 
        "normSD_RAVLT"
    ] = 1

    filtered_df.loc[
        (filtered_df['age_group'] == age_group) # because we already know all superagers are above the norm
        & (filtered_df["superager"] == 1),
        "normSD_RAVLT"
    ] = 1

# Display a random sample of 10 rows
sample_df = filtered_df.sample(10)

# Display the relevant columns
relevant_columns = ["id", "age", "YoE", "w1_delayed_recall_raw", "normSD_RAVLT"]
sample_df[relevant_columns]

In [ ]:
# Create a new valiable for those who fit the norm on all the tests and can be included in analysis

filtered_df["norm_neuropsych"] = 0

mask = (
    (filtered_df["normSD_TMTB"] == 1)
    & (filtered_df["normSD_sf"] == 1)
    & (filtered_df["normSD_dsf"] == 1)
    & (filtered_df["normSD_TMTA"] == 1)
    & (filtered_df["normSD_dsb"] == 1)
    & (filtered_df["normSD_RAVLT"] == 1)
)
filtered_df.loc[mask, "norm_neuropsych"] = 1

# Print number of participants within the norm
count_norm_neuropsych = filtered_df[filtered_df["norm_neuropsych"] == 1]["id"].count()
print(f"Number of participants with norm_neuropsych data: {count_norm_neuropsych}")

filtered_df[
    [
        "id",
        "normSD_TMTB",
        "normSD_sf",
        "normSD_dsf",
        "normSD_TMTA",
        "normSD_dsb",
        "normSD_RAVLT",
        "norm_neuropsych",
    ]
]

In [ ]:
# Check whether any superagers do not have norm neuropsych data from the 1.5SD analysis

superger_with_non_normal_neuropsych = filtered_df[
    (filtered_df["superager"] == 1) & (filtered_df["norm_neuropsych"] == 0)
]
superger_with_non_normal_neuropsych[
    [
        "id",
        "normSD_TMTB",
        "normSD_sf",
        "normSD_dsf",
        "normSD_TMTA",
        "normSD_dsb",
    ]
].head(50)

In [ ]:
# Create a new df 
clean_df = filtered_df

# Create a new variable with the superagers and those that meet the norm without dropping the others 
conditions = [
    (clean_df['superager'] == 1) & (clean_df['norm_neuropsych'] == 1),
    (clean_df['superager'] == 0) & (clean_df['norm_neuropsych'] == 1)
]

choices = [1, 0]

clean_df['sa_all'] = np.select(conditions, choices, default=2)

row_count = len(clean_df)
superager_count = clean_df["superager"].sum()
age_matched_controls = row_count - superager_count

print("Note that this is the number of participants who fit the classification for inclusion in the superager analysis at tp1 and have tp2 data available")
print(" ")
print(f"Number of participants: {row_count}")
print(f"Number of superagers: {superager_count:.0f}")
print(f"Number of age-matched controls: {age_matched_controls:.0f}")

# Get basic info about superagers
superager_df = clean_df[clean_df["sa_all"] == 1]

average_age = superager_df["age"].mean()
standard_deviation = superager_df["age"].std()

print(f"Average superager age: {average_age:.2f}")
print(f"Standard deviation: {standard_deviation:.2f}")

# Get basic info about controls
controls_df = clean_df[clean_df["sa_all"] == 0]
average_age = controls_df["age"].mean()
standard_deviation = controls_df["age"].std()

print(f"Average control age: {average_age:.2f}")
print(f"Standard deviation: {standard_deviation:.2f}")

clean_df[["id", "sa_all", "superager", "norm_neuropsych", "age", "YoE"]].head(15)

In [ ]:
# Export this df to a csv to use for future analysis

clean_df.to_csv(
    "/Users/rachelmorse/Documents/2023:2024/Data/BBHI/Exported data/superager_tp1.csv", index=False
)

<a id='timepoint2'></a>
### Timepoint 2

Now this section calculates who fit the classification for inclusion in the superager analysis at **timepoint 2** (whereas above is calculating who fit the classification for inclusion in the superager analysis at timepoint 1 even though all the participants have available timepoint 2 data).

In [ ]:
# Start by recreating filtered_df with tp2 data
# This needs to be done again to be able to use the ages of participants at tp2 

# Merge dfs by participant ID for main data and education 
merged_df_tp2 = pd.merge(m_df, df_edu, on="id", how="inner")

# Merge dfs by participant ID for merged data and MRI timept 1
merged_df_tp2 = pd.merge(merged_df_tp2, df_MRI1, on="id", how="inner")

# Merge dfs by participant ID for merged data and MRI timept 2
merged_df_tp2 = pd.merge(merged_df_tp2, df_MRI2, on="id", how="inner")

# Merge dfs by participant ID for merged data and neuropsych tp2 
duplicate_df = pd.merge(merged_df_tp2, np_tp2_df, on="id", how="inner")

# Print the number of participants after merging
print(f"Number of participants: {len(duplicate_df)}")

In [ ]:
# Calculate ages at tp2 assessment

def calculate_age(birth_date: str, collection_date: str) -> int:
    """Calculate age at assessment.

    Args:
        birth_date (str): Date of birth in format mm/dd/yyyy
        collection_date (str): Date of data collection in format mm/dd/yyyy

    Returns:
        age (int): Age at assessment
    """
    birth_date = datetime.strptime(birth_date, "%m/%d/%Y")
    collection_date = datetime.strptime(collection_date, "%m/%d/%Y")

    age = collection_date.year - birth_date.year

    # Check if the birthday has occurred for the current year
    if (collection_date.month, collection_date.day) < (
        birth_date.month,
        birth_date.day,
    ):
        age -= 1

    return age


# 'Q1 GNRL Age' is DOB and 'nps_date' is data collection date
# adds new column to df with age
duplicate_df["age_tp2"] = duplicate_df.apply(
    lambda row: calculate_age(row["Q1 GNRL Age"], row["nps_date"]), axis=1
)

# manually check that everything is running correctly
duplicate_df[["id", "age_tp2", "Q1 GNRL Age", "nps_date"]].head(5)

In [ ]:
# Filter df to only include 60+yrs at tp2 

# Print the number of participants before filtering
print(f"Number of participants before filtering: {len(duplicate_df)}")

duplicate_df = duplicate_df[duplicate_df["age_tp2"] >= 60]

# Print the number of participants after filtering
print(f"Number of participants after filtering: {len(duplicate_df)}")

duplicate_df[["id", "age_tp2"]].head(5)

In [ ]:
# Filter df to only include participants with MRI data

# Print the number of participants before filtering
print(f"Number of participants before filtering: {len(duplicate_df)}")

duplicate_df = duplicate_df[(duplicate_df["mri_data"] == 1) & (duplicate_df["mri_data_tp2"] == 1)]

# Print the number of participants after filtering
print(f"Number of participants after filtering: {len(duplicate_df)}")

duplicate_df[["id", "mri_data", "mri_data_tp2"]].head(5)

In [ ]:
# Drop participants with missing neuropsych data and -1 values in the specific columns

# Get the number of rows before dropping participants
rows_before = duplicate_df.shape[0]

columns = [
    "age_tp2",  # Age tp2
    "YoE",  # Years of education
    "w1_delayed_recall_raw",  # RAVLT tp1
    "w1_sem_fluency_raw",  # Semantic fluency tp1
    "w1_tmt_b_raw",  # TMT B tp1
    "w1_tmt_a_raw",  # TMT A tp1
    "w1_direct_digits_raw",  # Digit span forward tp1
    "w1_inverse_digits_raw",  # Digit span backward tp1
    "delayed_recall_raw",  # RAVLT tp2 
    "sem_fluency_raw",  # Semantic fluency tp2 
    "tmt_b_raw",  # TMT B tp2 
    "tmt_a_raw",  # TMT A tp2 
    "direct_digits_raw",  # Digit span forward tp2 
    "inverse_digits_raw",  # Digit span backward tp2 
]

# Drop participants with missing neuropsych data
duplicate_df.dropna(subset=columns, inplace=True)

# Drop participants with -1 values in specific columns
for col in columns:
    duplicate_df = duplicate_df[duplicate_df[col] != -1]

# Get the number of rows after dropping participants
rows_after = duplicate_df.shape[0]

# Calculate the number of participants dropped
participants_dropped = rows_before - rows_after

participants_dropped

In [ ]:
# Look at data for available longitudinal analysis timept 2

participants = duplicate_df["age_tp2"].count()
mean_age = duplicate_df["age_tp2"].mean()
std_dev_age = duplicate_df["age_tp2"].std()
min_age = duplicate_df["age_tp2"].min()
max_age = duplicate_df["age_tp2"].max()

print(f"Number of participants: {participants}")
print(f"Mean age: {mean_age:.2f}")
print(f"SD age: {std_dev_age:.2f}")
print(f"Range age: {min_age:.2f} - {max_age:.2f}")

In [ ]:
# Round down years of education data because some participants have data that is not a whole number (e.g. YoE = 9.5)
# This gives participants the lower education value because they will have their cognitive test scores normalized according to education level, disadvantaging them if it is rounded up

duplicate_df["YoE"] = np.floor(duplicate_df["YoE"])

In [ ]:
# Create scaled scores for TMT-A tp2 that adjust for age

# First round the raw scores to the nearest whole number (this is needed to use the neuronorma data)
duplicate_df["tmt_a_raw"] = duplicate_df["tmt_a_raw"].round()


def map_raw_to_scaled(raw_score, age):
    """Maps raw TMT-A scores to scaled TMT-A scores based on age.

    Args:
        raw_score (int): Raw TMT-A score from duplicate_df
        age (int): Age of participant
        TMTA (int): Raw TMT-A score from Neuronorma data
        Scale Score (int): Scaled TMT-A score from Neuronorma data

    Returns:
        TMTA_norm_age (int): Scaled TMT-A score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["TMTA"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


duplicate_df["TMTA_norm_tp2_age"] = duplicate_df.apply(
    lambda row: map_raw_to_scaled(row["tmt_a_raw"], row["age_tp2"]), axis=1
)

duplicate_df[["id", "age_tp2", "tmt_a_raw", "TMTA_norm_tp2_age"]].head(10)

In [ ]:
# Create scaled scores for TMT-A that adjust now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="TMTA",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
TMTA_norm2_list = []

for _, row in duplicate_df.iterrows():
    tmta_norm2_age = row["TMTA_norm_tp2_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    TMTA_norm2 = df_edu_data.loc[tmta_norm2_age, YoE]
    TMTA_norm2_list.append(TMTA_norm2)

duplicate_df["TMTA_norm_tp2"] = TMTA_norm2_list

duplicate_df[["id", "YoE", "TMTA_norm_tp2_age", "TMTA_norm_tp2"]].head(10)

In [ ]:
# Create scaled scores for TMT-B tp2 that adjust for age

# First round the raw scores to the nearest whole number (this is needed to use the neuronorma data)
duplicate_df["tmt_b_raw"] = duplicate_df["tmt_b_raw"].round()


def map_raw_to_scaled(raw_score, age):
    """Maps raw TMT-B scores to scaled TMT-B scores based on age.

    Args:
        raw_score (int): Raw TMT-B score from duplicate_df
        age (int): Age of participant
        TMTB (int): Raw TMT-B score from Neuronorma data
        Scale Score (int): Scaled TMT-B score from Neuronorma data

    Returns:
        TMTB_norm_age (int): Scaled TMT-B score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["TMTB"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


duplicate_df["TMTB_norm_tp2_age"] = duplicate_df.apply(
    lambda row: map_raw_to_scaled(row["tmt_b_raw"], row["age_tp2"]), axis=1
)

duplicate_df[["id", "age_tp2", "tmt_b_raw", "TMTB_norm_tp2_age"]].head(10)

In [ ]:
# Create scaled scores for TMT-B that adjust now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="TMTB",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
TMTB_norm2_list = []

for index, row in duplicate_df.iterrows():
    tmtb_norm2_age = row["TMTB_norm_tp2_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    TMTB_norm2 = df_edu_data.loc[tmtb_norm2_age, YoE]
    TMTB_norm2_list.append(TMTB_norm2)  # Corrected here

duplicate_df["TMTB_norm_tp2"] = TMTB_norm2_list

duplicate_df[["id", "YoE", "TMTB_norm_tp2_age", "TMTB_norm_tp2"]].head(10)

In [ ]:
# Create scaled scores for DS-Forward tp2 that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw DS-F scores to scaled DS-F scores based on age.

    Args:
        raw_score (int): Raw DS-F score from filtered_df
        age (int): Age of participant
        TMTA (int): Raw DS-F score from Neuronorma data
        Scale Score (int): Scaled DS-F score from Neuronorma data

    Returns:
        dsf_norm_age (int): Scaled DS-F score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["DS_F"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


duplicate_df["dsf_norm_tp2_age"] = duplicate_df.apply(
    lambda row: map_raw_to_scaled(row["direct_digits_raw"], row["age_tp2"]), axis=1
)

duplicate_df[["id", "age_tp2", "direct_digits_raw", "dsf_norm_tp2_age"]].head(10)

In [ ]:
# Create scaled scores for DS-Forward that adjust now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="DS_F",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
dsf_norm2_list = []

for index, row in duplicate_df.iterrows():
    dsf_norm2_age = row["dsf_norm_tp2_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    dsf_norm = df_edu_data.loc[dsf_norm2_age, YoE]
    dsf_norm2_list.append(dsf_norm)  # Corrected here

duplicate_df["dsf_norm_tp2"] = dsf_norm2_list

# Display df
duplicate_df[["id", "YoE", "dsf_norm_tp2_age", "dsf_norm_tp2"]].head(10)

In [ ]:
# Create scaled scores for for Digit Span - backward tp2 that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw DS-B scores to scaled DS-B scores based on age.

    Args:
        raw_score (int): Raw DS-B score from duplicate_df
        age (int): Age of participant
        TMTA (int): Raw DS-B score from Neuronorma data
        Scale Score (int): Scaled DS-B score from Neuronorma data

    Returns:
        dsf_norm_age (int): Scaled DS-B score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["DS_B"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


duplicate_df["dsb_norm_tp2_age"] = duplicate_df.apply(
    lambda row: map_raw_to_scaled(row["inverse_digits_raw"], row["age_tp2"]), axis=1
)

duplicate_df.sample(10)[["id", "age_tp2", "inverse_digits_raw", "dsb_norm_tp2_age"]].head(
    10
)

In [ ]:
# Create scaled scores for DS-Backward that adjust now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="DS_B",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
dsb_norm2_list = []

for index, row in duplicate_df.iterrows():
    dsb_norm2_age = row["dsb_norm_tp2_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    dsb_norm = df_edu_data.loc[dsb_norm2_age, YoE]
    dsb_norm2_list.append(dsb_norm)  # Corrected here

duplicate_df["dsb_norm_tp2"] = dsb_norm2_list

# Display a random 10 rows
duplicate_df.sample(10)[["id", "YoE", "dsb_norm_tp2_age", "dsb_norm_tp2"]]

In [ ]:
# Create scaled scores for Semantic Fluency tp2 that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw SF scores to scaled SF scores based on age.

    Args:
        raw_score (int): Raw SF score from duplicate_df
        age (int): Age of participant
        TMTA (int): Raw SF score from Neuronorma data
        Scale Score (int): Scaled SF score from Neuronorma data

    Returns:
        dsf_norm_age (int): Scaled SF score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["SF"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


duplicate_df["sf_norm_tp2_age"] = duplicate_df.apply(
    lambda row: map_raw_to_scaled(row["sem_fluency_raw"], row["age_tp2"]), axis=1
)

duplicate_df.sample(10)[["id", "age_tp2", "sem_fluency_raw", "sf_norm_tp2_age"]].head(10)

In [ ]:
# Create scaled scores for semantic fluency that adjust now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/Documents/2023:2024/Data/Neuronorma edu.xlsx",
    sheet_name="SF",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
sf_norm2_list = []

for index, row in duplicate_df.iterrows():
    sf_norm2_age = row["sf_norm_tp2_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    sf_norm = df_edu_data.loc[sf_norm2_age, YoE]
    sf_norm2_list.append(sf_norm)  # Corrected here

duplicate_df["sf_norm_tp2"] = sf_norm2_list

# Display a random 10 rows
duplicate_df.sample(10)[["id", "YoE", "sf_norm_tp2_age", "sf_norm_tp2"]]

In [ ]:
# Calculate who is superager based off of RAVLT score

# Schmidt 1996 - age 16-29 RAVLT-Delayed recall is scoring 12+ (no data adjusted by sex)

duplicate_df.loc[duplicate_df["delayed_recall_raw"] >= 12, "superager_RAVLT_tp2"] = 1
duplicate_df.loc[duplicate_df["delayed_recall_raw"] < 12, "superager_RAVLT_tp2"] = 0

print("Number of superagers meeting RAVLT criteria:")
print(duplicate_df[duplicate_df["superager_RAVLT_tp2"] == 1]["id"].count())

# manually check that everything is running correctly
duplicate_df.sample(10)[["id", "delayed_recall_raw", "superager_RAVLT_tp2"]]

In [ ]:
# Create new variables that = 1 when superagers are above 1SD below the norm for TMT-B and meet the RAVLT criteria

# Define the variables and the lower bound
variables = ["TMTA_norm_tp2"]
lower_bound = 10 - 1 * 3  # Scaled score of 10 is the mean and SD is 3 for all variables

# Create new column 'superager' and initialize it to 0
duplicate_df["superager_tp2"] = 0

# Update 'superager' to 1 for participants who score above the lower bound for all variables and have 1 for 'superager_RAVLT'
duplicate_df.loc[
    (duplicate_df[variables] >= lower_bound).all(axis=1) & (duplicate_df["superager_RAVLT_tp2"] == 1), 
    "superager_tp2"
] = 1

# Display a random sample of 10 rows
sample_df = duplicate_df.sample(10)

print(duplicate_df[duplicate_df["superager_tp2"] == 1]["id"].count())

# Display the relevant columns
relevant_columns = ["id", "age_tp2", "YoE","superager_tp2"]
sample_df[relevant_columns]


In [ ]:
# Create new variables that = 1 when non-superager participants are within 1.5SD of the norm and superager participants are above 1.5SD below the norm

variables = ["sf", "dsf", "TMTA", "TMTB", "dsb"]

for var in variables:
    # Calculate lower and upper bounds for each variable
    lower_bound = 10 - 1.5 * 3  # Scaled score of 10 is the mean and SD is 3 for all variables
    upper_bound = 10 + 1.5 * 3

    # Create new column 'normSD_x' where x is the variable
    duplicate_df[f"normSD_{var}_tp2"] = 0

    # Update 'normSD_x' to 1 if all participants are within the bounds for all the relevant variables and are not superagers
    duplicate_df.loc[
        (duplicate_df[f"{var}_norm_tp2"] >= lower_bound) 
        & (duplicate_df[f"{var}_norm_tp2"] <= upper_bound)
        & (duplicate_df["superager_tp2"] == 0), 
        f"normSD_{var}_tp2"
    ] = 1

    # Update 'normSD_x' to 1 if all participants meet the lower bound for all the relevant variables and are superagers
    duplicate_df.loc[
        (duplicate_df[f"{var}_norm_tp2"] >= lower_bound)
        & (duplicate_df["superager_tp2"] == 1), 
        f"normSD_{var}_tp2"
    ] = 1

# Display a random sample of 10 rows
sample_df = duplicate_df.sample(10)

# Display the relevant columns
relevant_columns = ["id", "age_tp2", "YoE"] + [f"normSD_{var}_tp2" for var in variables]
sample_df[relevant_columns]

In [ ]:
# Calculate who is within the norm for RAVLT

# Define the means and standard deviations for each age group
mean_sd = {
    '60-69': {'mean': 8.8, 'sd': 3.0},  # mean and SD for age group 60-69
    '70+': {'mean': 7.0, 'sd': 2.4}  # mean and SD for age group 70+
}

# Create a new column 'age_group'
duplicate_df['age_group'] = pd.cut(duplicate_df['age_tp2'], bins=[59, 69, np.inf], labels=['60-69', '70+'])

# Create new column 'normSD_var'
duplicate_df["normSD_RAVLT_tp2"] = 0

# For each age group, update 'normSD_var' based on the bounds for that group
for age_group, params in mean_sd.items():
    lower_bound = params['mean'] - 1.5 * params['sd']
    upper_bound = params['mean'] + 1.5 * params['sd']

    duplicate_df.loc[
        (duplicate_df['age_group'] == age_group) 
        & (duplicate_df['delayed_recall_raw'] >= lower_bound) 
        & (duplicate_df['delayed_recall_raw'] <= upper_bound)
        & (duplicate_df["superager_tp2"] == 0), 
        "normSD_RAVLT_tp2"
    ] = 1

    duplicate_df.loc[
        (duplicate_df['age_group'] == age_group) # Superagers are automatically assigned a 1 because we already know they meet the RAVLT criteria
        & (duplicate_df["superager_tp2"] == 1),
        "normSD_RAVLT_tp2"
    ] = 1

# Display a random sample of 10 rows
sample_df = duplicate_df.sample(10)

# Display the relevant columns
relevant_columns = ["id", "age_tp2", "YoE", "delayed_recall_raw", "normSD_RAVLT_tp2"]
sample_df[relevant_columns]

In [ ]:
# Create a new valiable for those who fit the norm on all the tests and can be included in analysis

duplicate_df["norm_neuropsych_tp2"] = 0

mask = (
    (duplicate_df["normSD_TMTB_tp2"] == 1)
    & (duplicate_df["normSD_sf_tp2"] == 1)
    & (duplicate_df["normSD_dsf_tp2"] == 1)
    & (duplicate_df["normSD_TMTA_tp2"] == 1)
    & (duplicate_df["normSD_dsb_tp2"] == 1)
    & (duplicate_df["normSD_RAVLT_tp2"] == 1)
)
duplicate_df.loc[mask, "norm_neuropsych_tp2"] = 1

# print number of participants within the norm
count_norm_neuropsych = duplicate_df[duplicate_df["norm_neuropsych_tp2"] == 1][
    "id"
].count()
print(f"Number of participants with norm_neuropsych_tp2 data: {count_norm_neuropsych}")

duplicate_df[
    [
        "id",
        "normSD_TMTB_tp2",
        "normSD_sf_tp2",
        "normSD_dsf_tp2",
        "normSD_TMTA_tp2",
        "normSD_dsb_tp2",
        "normSD_RAVLT_tp2",
        "norm_neuropsych_tp2",
    ]
]

In [ ]:
# Check whether any superagers do not have norm neuropsych data from the 1.5SD analysis

superger_with_non_normal_neuropsych_tp2 = duplicate_df[
    (duplicate_df["superager_tp2"] == 1) & (duplicate_df["norm_neuropsych_tp2"] == 0)
]
superger_with_non_normal_neuropsych_tp2[
    [
        "id",
        "normSD_TMTB_tp2",
        "normSD_sf_tp2",
        "normSD_dsf_tp2",
        "normSD_TMTA_tp2",
        "normSD_dsb_tp2",
    ]
].head(50)

In [ ]:
# Create a new df for tp2

clean_df_tp2 = duplicate_df

# Create a new variable with the superagers and those that meet the norm without dropping the others 
conditions = [
    (clean_df_tp2['superager_tp2'] == 1) & (clean_df_tp2['norm_neuropsych_tp2'] == 1),
    (clean_df_tp2['superager_tp2'] == 0) & (clean_df_tp2['norm_neuropsych_tp2'] == 1)
]

choices = [1, 0]

clean_df_tp2['sa_all_tp2'] = np.select(conditions, choices, default=2)


row_count = len(clean_df_tp2)
superager_count = clean_df_tp2["superager_tp2"].sum()
age_matched_controls = row_count - superager_count

print("Note that this is the number of participants who fit the classification for inclusion in the superager analysis at tp2 and have tp1 data available")
print(" ")
print(f"Number of participants: {row_count}")
print(f"Number of superagers at tp2: {superager_count:.0f}")
print(f"Number of age-matched controls at tp2: {age_matched_controls:.0f}")

# Get basic info about superagers
superager_df = clean_df_tp2[clean_df_tp2["sa_all_tp2"] == 1]

average_age = superager_df["age_tp2"].mean()
standard_deviation = superager_df["age_tp2"].std()

print(f"Average superager age: {average_age:.2f}")
print(f"Standard deviation: {standard_deviation:.2f}")

# Get basic info about controls
controls_df = clean_df_tp2[clean_df_tp2["sa_all_tp2"] == 0]
average_age = controls_df["age_tp2"].mean()
standard_deviation = controls_df["age_tp2"].std()

print(f"Average control age: {average_age:.2f}")
print(f"Standard deviation: {standard_deviation:.2f}")

clean_df_tp2[["id","sa_all_tp2", "superager_tp2", "norm_neuropsych_tp2", "age_tp2", "YoE"]].head(10)

Note that the number of participants in the timept 2 superager dataset (n=133) is larger than the timept 1 dataset (n=92) because there are more people who are aged 60+ years. The dataset of people who are within 1.5SD of the norm at timept 1 and 2 is even smaller than both of these as some people are within the norm at only one timepoint (this also excludes all people younger than 60 at tp1 who may have been older than 60 at tp2)

In [ ]:
# Export this df to a csv to use for future analysis

clean_df_tp2.to_csv(
    "/Users/rachelmorse/Documents/2023:2024/Data/BBHI/Exported data/superager_tp2.csv", index=False
)

<a id='timepoint1and2'></a>
### Timepoint 1 and 2

The following section merges the two data frames for timept 1 longitidinal data and timept 2 longitudinal data to create a new dataframe of participants who fit the norm at both timept 1 and 2. This is to compare superager status over time.

In [ ]:
# Merge the clean tp1 and tp2 dfs by participant ID - drop participants who are not in both dfs (e.g. those who do not have norm neuropsych data at tp1 or tp2)

# Note that the neuropsych variables for tp1 start with w1 (e.g. w1_tmt_a_raw) and the superager variables for tp2 have tp2 in the name (e.g. superager_tp2)

# Get the IDs from both dataframes before the merge
ids_before_merge = pd.concat([clean_df["id"], clean_df_tp2["id"]]).unique()

# Perform the merge
mergedtp_df = pd.merge(clean_df, clean_df_tp2, on="id", how="inner")

# Get the IDs after the merge
ids_after_merge = mergedtp_df["id"].unique()

# Find the IDs that were dropped during the merge
dropped_ids = set(ids_before_merge) - set(ids_after_merge)

# Print the dropped IDs (e.g. those who do not have norm neuropsych data at tp1 or tp2)
print(f"Dropped IDs: {dropped_ids}")

In [ ]:
# Merge filtered_df and duplicate_df to be able to see the tp1 and tp2 norm neuropsych data (including participants with 0 for norm neuropsych data)

# filtered_df has all participants with tp1 and tp2 data before excluding those without norm neuropsych data and has tp1 neuropsych data
# duplicate_df is a duplicate of filtered_df but then it has new variables including tp2 norm neuropsych data

# Perform the merge
merged_nonSA_df = pd.merge(filtered_df, duplicate_df, on="id", how="inner")

In [ ]:
# Filter the merged df to only include rows with IDs that were dropped when merging the clean tp1 and tp2 dfs

# This is just to check that the people being dropped are the ones who do not have norm neuropsych data at tp1 or tp2

dropped_rows_nonSA = merged_nonSA_df[merged_nonSA_df["id"].isin(dropped_ids)]

# Filter df further to only include rows where both norm_neuropsych and norm_neuropsych_tp2 are 1
filtered_dropped_rows_nonSA = dropped_rows_nonSA[
    (dropped_rows_nonSA["norm_neuropsych"] == 1)
    & (dropped_rows_nonSA["norm_neuropsych_tp2"] == 1)
]

# Display the result
filtered_dropped_rows_nonSA[["id", "norm_neuropsych", "norm_neuropsych_tp2"]]

# Becuase there are no participants in the result, we know that the participants who were dropped when merging the clean tp1 and tp2 dfs are the ones who do not have norm neuropsych data at tp1 or tp2

In [ ]:
# Now check whether any superagers were dropped when merging the clean tp1 and tp2 dfs

# This would mean that someone who was a superager in the clean data from tp1 or tp2 did not have norm neuropsych data at the other timept

dropped_rows_SA = mergedtp_df[mergedtp_df["id"].isin(dropped_ids)]

filtered_dropped_rows_SA = dropped_rows_SA[
    (dropped_rows_SA["superager"] == 1) | (dropped_rows_SA["superager_tp2"] == 1)
]

# Display the result
filtered_dropped_rows_SA[["id", "superager", "superager_tp2"]]

# Becuase there are no participants in the result, we know that no superagers were dropped when merging the clean tp1 and tp2 dfs

In [ ]:
# Get basic info about the merged clean df (e.g., with participants who have norm neuropsych data at tp1 and tp2)

row_count = len(mergedtp_df)
superager_tp1_count = mergedtp_df["superager"].sum()
age_matched_controls_tp1 = row_count - superager_tp1_count
superager_tp2_count = mergedtp_df["superager_tp2"].sum()
age_matched_controls_tp2 = row_count - superager_tp2_count

print("Note that this is the number of participants who fit the classification for inclusion in the superager analysis at tp1 and tp2")
print(" ")
print(f"Number of participants: {row_count}")
print(f"Number of superagers at tp1: {superager_tp1_count:.0f}")
print(f"Number of age-matched controls at tp1: {age_matched_controls_tp1:.0f}")
print(f"Number of superagers at tp2: {superager_tp2_count:.0f}")
print(f"Number of age-matched controls at tp2: {age_matched_controls_tp2:.0f}")

mergedtp_df[
    ["id", "superager", "norm_neuropsych", "superager_tp2", "norm_neuropsych_tp2"]
].head(10)

In [ ]:
# Define who is an SA at tp1 and tp2 

# Create a new variable 'superager_tp1tp2' which assigns 1 for SAs that maintain their status from tp1 to tp2
mergedtp_df['superager_tp1tp2'] = np.where((mergedtp_df['superager'] == 1) & (mergedtp_df['superager_tp2'] == 1), 1, 0)

superager_tp1tp2_count = mergedtp_df["superager_tp1tp2"].sum()

print(f"Number of superagers at tp1 who remain superagers at tp2: {superager_tp1tp2_count:.0f}")

In [ ]:
# Create a sankey diagram to show whether superager status changes between tp 1 and tp2
import plotly.graph_objects as go

# Count the number of people for each transition (e.g., from being a superager at tp1 to not being a superager at tp2)
superager_to_superager = (
    (mergedtp_df["superager"] == True) & (mergedtp_df["superager_tp2"] == True)
).sum()
superager_to_notsuperager = (
    (mergedtp_df["superager"] == True) & (mergedtp_df["superager_tp2"] == False)
).sum()
notsuperager_to_superager = (
    (mergedtp_df["superager"] == False) & (mergedtp_df["superager_tp2"] == True)
).sum()
notsuperager_to_notsuperager = (
    (mergedtp_df["superager"] == False) & (mergedtp_df["superager_tp2"] == False)
).sum()

# Define labels, source, target, and value
labels = ["Superager tp1", "Not Superager tp1", "Superager tp2", "Not Superager tp2"]
source = [0, 0, 1, 1]  # indices correspond to labels
target = [2, 3, 2, 3]
value = [
    superager_to_superager,
    superager_to_notsuperager,
    notsuperager_to_superager,
    notsuperager_to_notsuperager,
]

# Define colors for the nodes
colors = ["blue", "red", "blue", "red"]

# Create hover text
hovertext = [f"{v} people" for v in value]

# Create the Sankey diagram
fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(
                pad=15,
                thickness=20,
                line=dict(color="black", width=0.5),
                label=labels,
                color=colors,
            ),
            link=dict(
                source=source, target=target, value=value, hovertemplate=hovertext
            ),
        )
    ]
)

fig.update_layout(title_text="Superager transition from tp1 to tp2", font_size=10)
fig.show()

In [ ]:
# Take a look at the trajectory of episodic memory for the four groups shown in the Sankey diagram above

# Create a mapping dictionary for the group labels
group_labels = {
    '0.0-0.0': 'nonSA to nonSA',
    '0.0-1.0': 'nonSA to SA',
    '1.0-0.0': 'SA to nonSA',
    '1.0-1.0': 'SA to SA'
}

# Create a new column 'group' that represents the four groups
mergedtp_df['group'] = mergedtp_df['superager_RAVLT'].astype(str) + '-' + mergedtp_df['superager_RAVLT_tp2'].astype(str)

# Apply the mapping to the 'group' column
mergedtp_df['group'] = mergedtp_df['group'].map(group_labels)

# Reshape the DataFrame
long_df = pd.melt(mergedtp_df, id_vars=['id', 'group'], value_vars=['w1_delayed_recall_raw_x', 'delayed_recall_raw_y'], var_name='timepoint', value_name='RAVLT_score')


fig, axs = plt.subplots(2, 2, figsize=(10, 10))

groups = long_df.groupby('group')

# Determine the global min and max for the y-axis
global_min = long_df['RAVLT_score'].min()
global_max = long_df['RAVLT_score'].max()

for (name, group), ax in zip(groups, axs.flatten()):
    for key, grp in group.groupby('id'):
        grp.plot(ax=ax, kind='line', x='timepoint', y='RAVLT_score', legend=False)
    num_participants = group['id'].nunique()  # Number of unique participants in the group
    ax.set_title(f'Group: {name} (n={num_participants})')  # Add the number of participants to the title
    ax.set_ylim([global_min, global_max])  # Set the same y-axis limits for all subplots
    ax.set_xlabel('Timepoint')  # Add label to x-axis
    ax.set_ylabel('RAVLT Score')  # Add label to y-axis
    ax.set_xticks([0, 1])  # Set x-ticks
    ax.set_xticklabels(['tp1', 'tp2'])  # Set x-tick labels for the two tps

plt.tight_layout()
plt.show()

In [ ]:
# Create a new column 'SA_tp1_tp2' that concatenates 'superager_tp1' and 'superager_tp2'
mergedtp_df['SA_tp1_tp2'] = mergedtp_df['superager_RAVLT'].astype(str) + '-' + mergedtp_df['superager_RAVLT_tp2'].astype(str)

# Create a mapping dictionary for the SA change
SA_change_mapping = {
    '0.0-0.0': 0,  # nonSA to nonSA
    '0.0-1.0': 1,  # nonSA to SA
    '1.0-0.0': 2,  # SA to nonSA
    '1.0-1.0': 3   # SA to SA
}

# Create a new column 'SA_change' that represents the SA change
mergedtp_df['SA_change'] = mergedtp_df['SA_tp1_tp2'].map(SA_change_mapping)

In [ ]:
# Create a new column 'RAVLT_change_group' to categorize the participants
mergedtp_df['RAVLT_change_group'] = np.where(mergedtp_df['w1_delayed_recall_raw_x'] < mergedtp_df['delayed_recall_raw_y'], 'Increased',
                                             np.where(mergedtp_df['w1_delayed_recall_raw_x'] > mergedtp_df['delayed_recall_raw_y'], 'Decreased', 'Same'))

# Create a mapping dictionary for the group colors
group_colors = {
    '0.0-0.0': 'blue',  # nonSA to nonSA
    '0.0-1.0': 'green',  # nonSA to SA
    '1.0-0.0': 'red',  # SA to nonSA
    '1.0-1.0': 'purple'  # SA to SA
}

# Create a figure and axis
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Determine the global min and max for the y-axis
global_min = long_df['RAVLT_score'].min()
global_max = long_df['RAVLT_score'].max()

# For each change group, create a separate plot
for ax, (change_group, group_df) in zip(axs, mergedtp_df.groupby('RAVLT_change_group')):
    # For each participant in the change group, plot a line connecting their 'w1_delayed_recall_raw_x' and 'delayed_recall_raw_y' values
    for i in group_df.index:
        ax.plot(['tp1', 'tp2'], [group_df.loc[i, 'w1_delayed_recall_raw_x'], group_df.loc[i, 'delayed_recall_raw_y']], color=group_colors[group_df.loc[i, 'SA_tp1_tp2']])
    # Set the title and labels
    ax.set_title(f'RAVLT Scores for {change_group} Participants')
    ax.set_xlabel('Timepoint')
    ax.set_ylabel('RAVLT Score')
    ax.set_ylim([global_min, global_max])  # Set the same y-axis limits for all subplots


plt.tight_layout()
plt.show()